# PDF → Leitura em voz alta (TTS) com **pausas controladas** (Jupyter/Colab)

Este notebook faz:
1. Upload do PDF (Colab ou Jupyter com widget)
2. Extração de texto (PyMuPDF/pypdf)
3. (Opcional) OCR para PDF escaneado
4. Limpeza + normalização (evita leitura estranha de símbolos)
5. Segmentação por pontuação **(, ; : . ? !)**
6. Geração de áudio com `edge-tts` (sem SSML custom)
7. Merge final com **silêncio em ms por pontuação** (controle total das pausas)

> Dica: ajuste `PAUSE_MS` e `RATE` para ficar com “cara de audiolivro”.

In [ ]:
# ===== 1) Instalação de dependências =====
# Rode esta célula uma vez.
!pip -q install pymupdf pypdf edge-tts nest_asyncio ipywidgets

# Para MERGE em um único MP3, usaremos pydub + ffmpeg.
# No Colab, a célula abaixo instala ffmpeg automaticamente.

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.1/24.1 MB 29.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 329.6/329.6 kB 10.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 17.8 MB/s eta 0:00:00


In [ ]:
# ===== 2) (Recomendado) Instalar ffmpeg para juntar os áudios =====
# - Colab: instala via apt-get
# - Jupyter local: se der erro, instale ffmpeg no seu sistema (Windows/Linux/Mac) ou pule o MERGE.

import sys

def running_in_colab():
    return "google.colab" in sys.modules

if running_in_colab():
    !apt-get -y update >/dev/null
    !apt-get -y install ffmpeg >/dev/null
    print("ffmpeg instalado (Colab).")
else:
    print("Não-Colab: se o merge falhar, instale ffmpeg no seu sistema.")

!pip -q install pydub

In [ ]:
# ===== 3) Upload do PDF (Colab ou Jupyter) =====
from pathlib import Path
import sys

PDF_PATH = None

def running_in_colab():
    return "google.colab" in sys.modules

if running_in_colab():
    from google.colab import files
    up = files.upload()
    if not up:
        raise RuntimeError("Nenhum arquivo foi enviado.")
    fname = next(iter(up.keys()))
    PDF_PATH = str(Path(fname).resolve())
    print("Arquivo carregado:", PDF_PATH)
else:
    # Jupyter local: tenta ipywidgets FileUpload
    try:
        import ipywidgets as widgets
        from IPython.display import display, clear_output

        uploader = widgets.FileUpload(accept=".pdf", multiple=False)
        display(uploader)
        print("↑ Selecione um PDF acima e RE-EXECUTE esta célula para salvar o arquivo.")

        if uploader.value:
            item = next(iter(uploader.value.values()))
            content = item.get("content", None)
            name = item.get("metadata", {}).get("name", "arquivo.pdf")

            if content is None:
                content = item
                name = getattr(item, "name", "arquivo.pdf")

            out = Path(name).name
            Path(out).write_bytes(content)
            PDF_PATH = str(Path(out).resolve())
            clear_output()
            print("Arquivo carregado:", PDF_PATH)
    except Exception as e:
        print("Não consegui ativar upload por widget aqui.")
        print("Alternativas:")
        print("1) Coloque o PDF na mesma pasta do notebook e defina PDF_PATH manualmente.")
        print("Erro:", repr(e))

# Se precisar setar manualmente, descomente e ajuste:
# PDF_PATH = r"caminho/para/seu_arquivo.pdf"

In [ ]:
# ===== 4) Extrair texto do PDF =====
import re
import pymupdf  # PyMuPDF
from pypdf import PdfReader

def extract_text_pymupdf(pdf_path: str) -> str:
    doc = pymupdf.open(pdf_path)
    parts = []
    for page in doc:
        parts.append(page.get_text("text") or "")
    return "\n".join(parts)

def extract_text_pypdf(pdf_path: str) -> str:
    reader = PdfReader(pdf_path)
    parts = []
    for page in reader.pages:
        parts.append(page.extract_text() or "")
    return "\n".join(parts)

def extract_text_best_effort(pdf_path: str) -> str:
    t1 = ""
    try:
        t1 = extract_text_pymupdf(pdf_path)
    except Exception as e:
        print("PyMuPDF falhou, tentando pypdf. Erro:", repr(e))

    t2 = ""
    try:
        t2 = extract_text_pypdf(pdf_path)
    except Exception as e:
        print("pypdf falhou. Erro:", repr(e))

    return t1 if len(t1) >= len(t2) else t2

if not PDF_PATH:
    raise RuntimeError("PDF_PATH está vazio. Volte na célula de upload e carregue um PDF.")

raw_text = extract_text_best_effort(PDF_PATH)
print("Caracteres extraídos:", len(raw_text))
print("\nAmostra:\n", raw_text[:1200])

In [ ]:
# ===== 5) (Opcional) OCR se o PDF for escaneado =====
# Se quase não saiu texto (ex.: < 500 caracteres), ative USE_OCR=True.
# OBS: OCR pode exigir instalação de Tesseract e Poppler no sistema.
#
# Colab (recomendado):
#   !apt-get -y update
#   !apt-get -y install -y tesseract-ocr poppler-utils
#   !pip -q install pdf2image pytesseract
#
USE_OCR = False

def needs_ocr(text: str, threshold: int = 500) -> bool:
    return (text is None) or (len(text.strip()) < threshold)

if USE_OCR or needs_ocr(raw_text):
    print("Texto parece insuficiente. Tentando OCR...")
    try:
        !pip -q install pdf2image pytesseract
        if running_in_colab():
            !apt-get -y install -y tesseract-ocr poppler-utils >/dev/null
        from pdf2image import convert_from_path
        import pytesseract

        def ocr_pdf(pdf_path: str, dpi: int = 200, max_pages=None) -> str:
            images = convert_from_path(pdf_path, dpi=dpi)
            if max_pages is not None:
                images = images[:max_pages]
            parts = []
            for i, img in enumerate(images, 1):
                txt = pytesseract.image_to_string(img, lang="por")
                parts.append(txt)
                print(f"OCR página {i}/{len(images)} OK (chars: {len(txt)})")
            return "\n".join(parts)

        # Dica: para testar rápido, limite páginas:
        # raw_text = ocr_pdf(PDF_PATH, dpi=200, max_pages=5)
        raw_text = ocr_pdf(PDF_PATH, dpi=200, max_pages=None)
        print("OCR concluído. Caracteres:", len(raw_text))
    except Exception as e:
        print("Falha no OCR. Prováveis causas: falta Tesseract/Poppler no sistema.")
        print("Erro:", repr(e))
else:
    print("OCR não necessário (ou USE_OCR=False).")

In [ ]:
# ===== 6) Limpeza + normalização para TTS =====
# Objetivo: melhorar fluidez, manter parágrafos e evitar pronúncia “estranha” de símbolos.

PARA_TOKEN = "<PARA>"

def clean_text_basic(t: str) -> str:
    t = (t or "").replace("\x00", " ")
    # remove hifenização por quebra de linha: "exem-\nplo" -> "exemplo"
    t = re.sub(r"(\w)-\n(\w)", r"\1\2", t)
    # normaliza quebras e espaços
    t = re.sub(r"\r\n", "\n", t)
    # preserva parágrafos (dupla quebra) para pausas maiores
    t = re.sub(r"\n{2,}", "\n\n", t)
    t = re.sub(r"[ \t]+", " ", t)
    # quebra simples vira espaço quando não é final de frase
    t = re.sub(r"(?<![.!?…])\n", " ", t)
    t = re.sub(r" {2,}", " ", t)
    return t.strip()

def normalize_for_tts(t: str) -> str:
    # bullets e travessões viram pausa de frase
    t = t.replace("•", ". ").replace("–", ". ").replace("—", ". ")
    # listas tipo " - item" viram frase (evita falar "hífen")
    t = re.sub(r"\s-\s", ". ", t)
    # símbolos comuns que costumam soar estranhos
    t = t.replace("§", " seção ")
    t = t.replace("%", " por cento ")
    # preserva parágrafos para pausa extra
    t = t.replace("\n\n", f" {PARA_TOKEN} ")
    # normaliza pontos repetidos
    t = re.sub(r"[.]{2,}", ".", t)
    t = re.sub(r"\s{2,}", " ", t)
    return t.strip()

text_clean = normalize_for_tts(clean_text_basic(raw_text))
print("Caracteres após limpeza:", len(text_clean))
print("\nAmostra:\n", text_clean[:1200])


In [ ]:
# ===== 7) Segmentação por pontuação (controle fino de pausas) =====
# Vamos quebrar o texto em segmentos que terminam em: , ; : . ? !
# Depois, no merge, inserimos silêncio baseado nessa pontuação.
# Também preservamos parágrafos para pausas maiores (audiobook).

# Ajuste de tamanho dos segmentos (audiobook tende a ficar mais natural em blocos médios)
TARGET_MIN_CHARS = 160
TARGET_MAX_CHARS = 360


def split_with_punct(text: str):
    text = (text or "").strip()
    if not text:
        return []

    # garante espaço depois de pontuação (ajuda o split)
    text = re.sub(r"([,;:\.\?!])(?=\S)", r"\1 ", text)
    text = text.replace(PARA_TOKEN, f" {PARA_TOKEN} ")

    paragraphs = [p.strip() for p in text.split(PARA_TOKEN)]

    out = []
    for idx, para in enumerate(paragraphs):
        if not para:
            continue
        parts = re.findall(r".+?[\,\.;:\?!](?:\s+|$)|.+$", para, flags=re.DOTALL)
        for p in parts:
            p = p.strip()
            if not p:
                continue
            punct = p[-1] if p[-1] in ",;:.?!" else "."
            out.append((p, punct, False))
        # marca quebra de parágrafo no último segmento do parágrafo
        if idx < len(paragraphs) - 1 and out:
            last = out[-1]
            out[-1] = (last[0], last[1], True)
    return out


def merge_short_segments(segments, min_chars=TARGET_MIN_CHARS, max_chars=TARGET_MAX_CHARS):
    merged = []
    buffer = ""
    buffer_punct = "."
    buffer_para = False

    for txt, punct, para in segments:
        if not buffer:
            buffer = txt
            buffer_punct = punct
            buffer_para = para
        else:
            candidate = f"{buffer} {txt}".strip()
            if len(candidate) <= max_chars:
                buffer = candidate
                buffer_punct = punct
                buffer_para = para
            else:
                merged.append((buffer, buffer_punct, buffer_para))
                buffer = txt
                buffer_punct = punct
                buffer_para = para

        if len(buffer) >= min_chars and buffer_para:
            merged.append((buffer, buffer_punct, buffer_para))
            buffer = ""
            buffer_punct = "."
            buffer_para = False

    if buffer:
        merged.append((buffer, buffer_punct, buffer_para))

    return merged

segments = merge_short_segments(split_with_punct(text_clean))

print("Total de segmentos:", len(segments))
print("Exemplo 1:", segments[0][0][:180], "| punct:", segments[0][1] if segments else None)


In [ ]:
# ===== 8) Escolha de voz (robusta: não quebra se uma falhar) =====
import os, asyncio
import nest_asyncio
nest_asyncio.apply()
import edge_tts

from IPython.display import Audio, display

async def list_ptbr_voices():
    voices = await edge_tts.list_voices()
    ptbr = sorted({v["ShortName"] for v in voices if v.get("Locale") == "pt-BR"})
    return ptbr

def filter_multilingual(voices):
    return [v for v in voices if "Multilingual" not in v]

async def synth_one(text, voice, rate, pitch, out_path, retries=2):
    last_err = None
    for attempt in range(retries + 1):
        try:
            comm = edge_tts.Communicate(text=text, voice=voice, rate=rate, pitch=pitch)
            await comm.save(out_path)
            return True
        except Exception as e:
            last_err = e
            await asyncio.sleep(0.8 * (attempt + 1))
    print(f"Falhou {voice}: {type(last_err).__name__}: {last_err}")
    return False

ptbr = await list_ptbr_voices()
ptbr_non_multi = filter_multilingual(ptbr)

print("Vozes pt-BR disponíveis agora:", ptbr)
if ptbr_non_multi:
    print("Vozes pt-BR (sem Multilingual):", ptbr_non_multi)

SAMPLE_TEXT = (
    "Teste de voz. Pausas e naturalidade importam. "
    "Se estiver rápido demais, vou falar um pouco mais devagar, ok?"
)

SAMPLE_OUT = "samples_voz"
os.makedirs(SAMPLE_OUT, exist_ok=True)

# Ajuste aqui (mais lento costuma soar menos robótico)
TEST_RATE  = "-10%"
TEST_PITCH = "-2Hz"

# Testa até 5 vozes (preferindo não-multilingual)
VOICES_TO_TEST = (ptbr_non_multi or ptbr)[:5]

sample_files = []
for v in VOICES_TO_TEST:
    outp = os.path.join(SAMPLE_OUT, f"{v}.mp3")
    ok = await synth_one(SAMPLE_TEXT, v, TEST_RATE, TEST_PITCH, outp, retries=2)
    if ok:
        print("OK:", outp)
        sample_files.append(outp)

for f in sample_files:
    print(f)
    display(Audio(f, autoplay=False))


In [ ]:
# ===== 9) Gerar áudio por segmento (preferir voz pt-BR não-multilingual) =====

# Preferência por vozes sem "Multilingual" reduz variação de sotaque.
PREFERRED_VOICES = [
    "pt-BR-FranciscaNeural",
    "pt-BR-AntonioNeural",
    "pt-BR-ElzaNeural",
    "pt-BR-FabioNeural",
]

def choose_voice(available):
    for v in PREFERRED_VOICES:
        if v in available:
            return v
    return available[0] if available else "pt-BR-FranciscaNeural"

try:
    ptbr
except NameError:
    ptbr = await list_ptbr_voices()

ptbr_non_multi = filter_multilingual(ptbr)

VOICE = choose_voice(ptbr_non_multi or ptbr)
RATE  = "-10%"  # mais lento = mais natural (audiobook)
PITCH = "-2Hz"

OUT_DIR = "out_audio_segs"
os.makedirs(OUT_DIR, exist_ok=True)

# Para teste rápido, limite o número de segmentos (ex.: 200)
LIMIT = None  # ou 200

async def synthesize_segments(segments, out_dir=OUT_DIR, voice=VOICE, rate=RATE, pitch=PITCH, limit=None):
    files = []
    todo = segments if limit is None else segments[:limit]
    for i, (txt, punct, para_break) in enumerate(todo, 1):
        out_path = os.path.join(out_dir, f"seg_{i:05d}_{ord(punct)}.mp3")
        ok = await synth_one(txt, voice, rate, pitch, out_path, retries=2)
        if ok:
            files.append((out_path, punct, para_break))
        else:
            files.append((None, punct, para_break))
        if i % 50 == 0:
            print(f"{i}/{len(todo)}")
    return files

seg_files = await synthesize_segments(segments, limit=LIMIT)
print("Segmentos gerados (inclui falhas puladas):", len(seg_files))
print("Voz escolhida:", VOICE)


In [ ]:
# ===== 10) Merge final com pausas configuráveis por pontuação =====
from pydub import AudioSegment
from IPython.display import Audio, display
import os

# Ajuste as pausas aqui (em milissegundos)
PAUSE_MS = {
    ",": 240,
    ";": 480,
    ":": 480,
    ".": 520,
    "?": 560,
    "!": 560,
}

# Pausa adicional para quebra de parágrafo (audiobook)
PARA_BREAK_MS = 750

MERGE = True

if MERGE:
    combined = AudioSegment.empty()
    count_added = 0

    for path, punct, para_break in seg_files:
        if path and os.path.exists(path):
            combined += AudioSegment.from_file(path, format="mp3")
            pause = PAUSE_MS.get(punct, 320)
            if para_break:
                pause += PARA_BREAK_MS
            combined += AudioSegment.silent(duration=pause)
            count_added += 1

    final_path = "audio_final.mp3"
    combined.export(final_path, format="mp3")

    print("Segmentos adicionados:", count_added)
    print("Arquivo final:", final_path)
    display(Audio(final_path, autoplay=False))
else:
    print("MERGE=False. Os segmentos estão em:", OUT_DIR)


## Ajustes rápidos (como “tunar” o resultado)

- **Pausas:** edite `PAUSE_MS` (ms) na célula de merge.
  - Quer mais pausa na vírgula? aumente `","` para 300–450.
  - Quer pausa “dramática” em ponto final? aumente `"."` para 800–1100.

- **Naturalidade:** ajuste `RATE` e `VOICE`.
  - `RATE=-8%` ou `-10%` geralmente fica menos robótico.
  - Teste as vozes na célula “Escolha de voz”.

- **PDF escaneado:** ative `USE_OCR=True` (a célula de OCR explica dependências).

- **Teste rápido:** use `LIMIT=200` para não demorar num PDF grande.